In [5]:
import pandas as pd

expr_xena = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TCGA.BRCA.sampleMap_HiSeqV2.gz",
    sep="\t",
    index_col=0
)

expr_xena.shape


(20530, 1218)

In [7]:
import numpy as np

# expression threshold
expr_threshold = 1
min_samples = int(0.10 * expr_xena.shape[1])

# filter genes
expr_filtered = expr_xena[
    (expr_xena > expr_threshold).sum(axis=1) >= min_samples
]

expr_xena.shape, expr_filtered.shape


((20530, 1218), (18231, 1218))

In [9]:
# compute gene-wise variance
gene_var = expr_filtered.var(axis=1)

# variance cutoff (25th percentile)
var_cutoff = gene_var.quantile(0.25)

# apply variance filter
expr_var_filtered = expr_filtered[gene_var > var_cutoff]

expr_filtered.shape, expr_var_filtered.shape


((18231, 1218), (13673, 1218))

In [11]:
expr_var_filtered.shape


(13673, 1218)

# PAM50 subtype annotation

In [13]:
# load Xena phenotype (contains PAM50)
pheno = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TCGA.BRCA (2).sampleMap_BRCA_clinicalMatrix",
    sep="\t"
)

# keep only sample ID + PAM50
pam50 = pheno[["sampleID", "PAM50Call_RNAseq"]].dropna()
pam50 = pam50.set_index("sampleID")

# align samples
common_samples = expr_var_filtered.columns.intersection(pam50.index)

expr_final = expr_var_filtered[common_samples]
pam50_final = pam50.loc[common_samples]

# quick check
expr_final.shape, pam50_final["PAM50Call_RNAseq"].value_counts()


((13673, 956),
 PAM50Call_RNAseq
 LumA      434
 LumB      194
 Basal     142
 Normal    119
 Her2       67
 Name: count, dtype: int64)

# Start PHASE 5A

# Build PPI backbone

In [15]:
import pandas as pd
import gzip

# STRING protein info file (ID → gene symbol)
map_url = "https://stringdb-static.org/download/protein.info.v11.5/9606.protein.info.v11.5.txt.gz"

prot_info = pd.read_csv(
    map_url,
    sep="\t",
    compression="gzip"
)

prot_info.columns



Index(['#string_protein_id', 'preferred_name', 'protein_size', 'annotation'], dtype='object')

# Map STRING protein IDs → HUGO gene symbols

In [17]:
prot_info.columns


Index(['#string_protein_id', 'preferred_name', 'protein_size', 'annotation'], dtype='object')

In [23]:
import pandas as pd

ppi = pd.read_csv(
    "https://stringdb-static.org/download/protein.links.detailed.v11.5/9606.protein.links.detailed.v11.5.txt.gz",
    sep=" ",
    compression="gzip"
)

ppi_hq = ppi[ppi["combined_score"] >= 700]

ppi_hq.shape


(505968, 10)

In [25]:
prot_info = pd.read_csv(
    "https://stringdb-static.org/download/protein.info.v11.5/9606.protein.info.v11.5.txt.gz",
    sep="\t",
    compression="gzip"
)

prot_info.columns


Index(['#string_protein_id', 'preferred_name', 'protein_size', 'annotation'], dtype='object')

In [27]:
# rename STRING protein ID column
prot_info_fixed = prot_info.rename(
    columns={"#string_protein_id": "protein_id"}
)

# build protein → gene map
prot_map = prot_info_fixed[["protein_id", "preferred_name"]].drop_duplicates()

# map protein IDs in PPI to gene symbols
ppi_mapped = (
    ppi_hq
    .merge(prot_map, left_on="protein1", right_on="protein_id", how="left")
    .rename(columns={"preferred_name": "gene1"})
    .drop(columns=["protein_id"])
    .merge(prot_map, left_on="protein2", right_on="protein_id", how="left")
    .rename(columns={"preferred_name": "gene2"})
    .drop(columns=["protein_id"])
)

# keep only edges with both genes mapped
ppi_genes = ppi_mapped.dropna(subset=["gene1", "gene2"])

ppi_hq.shape, ppi_genes.shape


((505968, 10), (505968, 12))

In [29]:
ppi_genes[["gene1", "gene2"]].isna().sum()


gene1    0
gene2    0
dtype: int64

In [31]:
# genes expressed in TCGA-BRCA
expressed_genes = set(expr_final.index)

# keep PPI edges where both genes are expressed
ppi_tcga = ppi_genes[
    ppi_genes["gene1"].isin(expressed_genes) &
    ppi_genes["gene2"].isin(expressed_genes)
]

ppi_genes.shape, ppi_tcga.shape


((505968, 12), (194496, 12))

In [33]:
# save PPI backbone
ppi_tcga.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\PPI_TCGA_BRCA_STRING_HQ.csv",
    index=False
)

# basic network stats
num_nodes = len(set(ppi_tcga["gene1"]) | set(ppi_tcga["gene2"]))
num_edges = ppi_tcga.shape[0]

num_nodes, num_edges


(10095, 194496)

# 5.2= Create subtype-specific seed gene sets

In [35]:
# load Luminal A DEG table (already generated earlier)
deg_luma = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumA_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

# extract DEG gene symbols
luma_deg_genes = set(deg_luma.index)

len(luma_deg_genes)


870

In [37]:
import pandas as pd

# load the COSMIC CGC file (adjust extension if needed)
cgc = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Census_allWed Jan 28 09_32_24 2026.csv"
)

# basic inspection
cgc.shape, cgc.columns.tolist()[:10]


((763, 20),
 ['Gene Symbol',
  'Name',
  'Entrez GeneId',
  'Genome Location',
  'Tier',
  'Hallmark',
  'Chr Band',
  'Somatic',
  'Germline',
  'Tumour Types(Somatic)'])

In [39]:
cgc["Tier"].value_counts(dropna=False)


Tier
1    590
2    173
Name: count, dtype: int64

In [41]:
# correct Tier filtering (numeric, not string)
drivers_filt = cgc[cgc["Tier"].isin([1, 2])]

driver_genes = set(drivers_filt["Gene Symbol"].str.upper())

len(driver_genes)


763

# Merge LumA DEGs + drivers → restrict to PPI backbone

In [43]:
# merge LumA DEGs + COSMIC drivers
luma_seed_all = set(g.upper() for g in luma_deg_genes) | driver_genes

# restrict seeds to genes present in PPI backbone
ppi_nodes = set(ppi_tcga["gene1"]) | set(ppi_tcga["gene2"])

luma_seed_final = luma_seed_all.intersection(ppi_nodes)

len(luma_seed_all), len(luma_seed_final)


(1622, 1052)

In [47]:
pd.Series(sorted(luma_seed_final), name="gene").to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_LumA.csv",
    index=False
)


# Step 5.2 (Luminal B seeds)

In [45]:
# load Luminal B DEG table
deg_lumb = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_LumB_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

# extract gene symbols
lumb_deg_genes = set(deg_lumb.index)

len(lumb_deg_genes)


4682

In [49]:
# merge LumB DEGs + COSMIC drivers
lumb_seed_all = set(g.upper() for g in lumb_deg_genes) | driver_genes

len(lumb_seed_all)


5253

In [51]:
# restrict LumB seeds to PPI nodes
lumb_seed_final = lumb_seed_all.intersection(ppi_nodes)

len(lumb_seed_all), len(lumb_seed_final)


(5253, 3901)

In [53]:
pd.Series(sorted(lumb_seed_final), name="gene").to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_LumB.csv",
    index=False
)


# Step 5.2 (HER2 seeds)

In [55]:
# load HER2 DEG table
deg_her2 = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_HER2_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

# extract gene symbols
her2_deg_genes = set(deg_her2.index)

len(her2_deg_genes)


4684

In [57]:
# merge HER2 DEGs with driver genes
her2_seed_all = her2_deg_genes.union(driver_genes)

len(her2_seed_all)


5276

In [59]:
# restrict HER2 seeds to PPI nodes
her2_seed_final = her2_seed_all.intersection(ppi_nodes)

len(her2_seed_all), len(her2_seed_final)


(5276, 3883)

In [61]:
pd.Series(sorted(her2_seed_final), name="gene").to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_HER2.csv",
    index=False
)


# Step 5.2 (Basal / TNBC seeds)

In [63]:
import pandas as pd

deg_basal_final = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv",
    index_col=0
)

deg_basal_final.shape


(4857, 3)

In [65]:
# extract TNBC DEG genes
basal_deg_genes = set(deg_basal_final.index)

# merge TNBC DEGs + COSMIC drivers
basal_seed_all = set(g.upper() for g in basal_deg_genes) | driver_genes

len(basal_seed_all)


5442

In [67]:
# restrict TNBC seeds to PPI nodes
basal_seed_final = basal_seed_all.intersection(ppi_nodes)

len(basal_seed_all), len(basal_seed_final)


(5442, 3993)

In [69]:
pd.Series(sorted(basal_seed_final), name="gene").to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_TNBC.csv",
    index=False
)


# PHASE 5B — Network propagation (personalized PageRank)